# Task 2C: Evaluation and baseline comparison
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka/blob/main/task2_genai/Task2_Eval.ipynb)

Base model (same system prompt, no fine-tuning) vs merged fine-tuned model on the **identical** held-out test set, greedy decoding for both. Needs a T4 GPU and `GEMINI_API_KEY` (LLM judge: `gemini-3.1-flash-lite`).

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "task2_genai/requirements-train.txt"], check=True)
else:
    while not os.path.exists("requirements.txt"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("repo root:", os.getcwd(), "| colab:", IN_COLAB)

repo root: /content/repo | colab: True


In [2]:
# Colab ships torchao 0.10, which recent peft/transformers reject; this project does not use torchao
!pip uninstall -y -q torchao

In [3]:
import gc, json, torch, pandas as pd
from task2_genai.dataset import DATA_DIR, load_jsonl
from task2_genai.finetune import TrainConfig, free_gpu
from task2_genai.evaluation import generate, load_for_inference, prompt_messages

BASE_ID = TrainConfig().base_model
from common.config import get_secret
from huggingface_hub import HfApi
FT_ID = f"{HfApi(token=get_secret('HF_TOKEN')).whoami()['name']}/qwen2.5-3b-risk-extractor"  # pushed by the fine-tune notebook
test = load_jsonl(DATA_DIR / "test.jsonl")
os.makedirs("task2_genai/results", exist_ok=True)
print(len(test), "test examples")

32 test examples


In [4]:
def run(model_id, name):
    model, tok = load_for_inference(model_id)
    gens = generate(model, tok, [prompt_messages(r) for r in test])
    with open(f"task2_genai/results/{name}_generations.jsonl", "w", encoding="utf-8") as f:
        for r, g in zip(test, gens):
            f.write(json.dumps({"id": r["id"], **g}, ensure_ascii=False) + "\n")
    del model
    free_gpu()
    return gens

base_gens = run(BASE_ID, "base")
ft_gens = run(FT_ID, "finetuned")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.17GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## ROUGE-L and structural metrics

In [5]:
from task2_genai.evaluation import evaluate_model
base_summary, base_rows = evaluate_model("base (system prompt only)", test, base_gens)
ft_summary, ft_rows = evaluate_model("fine-tuned (QLoRA)", test, ft_gens)
comparison = pd.DataFrame([base_summary, ft_summary]).set_index("model").T
comparison["delta"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison.round(4)

model,base (system prompt only),fine-tuned (QLoRA),delta
rougeL_f1,0.6011,0.8823,0.2812
json_valid_rate,0.1250,1.0000,0.8750
category_precision,0.8182,0.9385,0.1203
category_recall,0.1385,0.9385,0.8000
category_f1,0.2368,0.9385,0.7016
exact_category_set,0.0312,0.9062,0.8750
quote_grounding_rate,0.6364,0.9846,0.3483
severity_accuracy,0.6667,0.7213,0.0546
auto_hallucination_rate,0.0938,0.0312,-0.0625


## Additional metrics: BERTScore F1 and LLM-as-judge (structured JSON rubric)

In [6]:
from task2_genai.evaluation import bertscore_f1, gold_of
refs = [json.dumps(gold_of(r), ensure_ascii=False) for r in test]
try:
    base_rows["bertscore_f1"] = bertscore_f1([g["text"] for g in base_gens], refs)
    ft_rows["bertscore_f1"] = bertscore_f1([g["text"] for g in ft_gens], refs)
    comparison.loc["bertscore_f1"] = [base_rows["bertscore_f1"].mean(), ft_rows["bertscore_f1"].mean(),
                                      ft_rows["bertscore_f1"].mean() - base_rows["bertscore_f1"].mean()]
except Exception as exc:  # library incompatibility must not block the LLM-judge metric below
    print(f"BERTScore unavailable: {type(exc).__name__}: {exc}")
comparison.round(4)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model,base (system prompt only),fine-tuned (QLoRA),delta
rougeL_f1,0.6011,0.8823,0.2812
json_valid_rate,0.1250,1.0000,0.8750
category_precision,0.8182,0.9385,0.1203
category_recall,0.1385,0.9385,0.8000
category_f1,0.2368,0.9385,0.7016
exact_category_set,0.0312,0.9062,0.8750
quote_grounding_rate,0.6364,0.9846,0.3483
severity_accuracy,0.6667,0.7213,0.0546
auto_hallucination_rate,0.0938,0.0312,-0.0625
bertscore_f1,0.8655,0.9823,0.1168


In [7]:
from task2_genai.evaluation import judge_all, JUDGE_MODEL
print("judge:", JUDGE_MODEL)
judge_base = judge_all(test, [g["text"] for g in base_gens])
judge_ft = judge_all(test, [g["text"] for g in ft_gens])
crit = ["faithfulness", "completeness", "severity_calibration", "format_compliance"]
judge_table = pd.DataFrame({"base": judge_base[crit].mean(), "fine-tuned": judge_ft[crit].mean()})
judge_table.loc["verdict=correct"] = [(judge_base.verdict == "correct").mean(), (judge_ft.verdict == "correct").mean()]
judge_table.loc["verdict=hallucinated"] = [(judge_base.verdict == "hallucinated").mean(), (judge_ft.verdict == "hallucinated").mean()]
judge_table.round(3)

judge: gemini:gemini-3.1-flash-lite


,base,fine-tuned
faithfulness,3.500,4.781
completeness,3.188,4.781
severity_calibration,3.781,4.281
format_compliance,3.125,5.000
verdict=correct,0.000,0.656
verdict=hallucinated,0.375,0.031


In [8]:
judge_ft.head(5)

,id,faithfulness,completeness,severity_calibration,format_compliance,verdict,rationale
0,ex_0317,5,5,5,5,correct,The candidate extraction accurately identifies...
1,ex_0243,5,5,4,5,correct,The candidate extraction is faithful to the te...
2,ex_0042,5,4,3,5,partially_correct,The candidate extraction is faithful to the te...
3,ex_0171,5,5,5,5,correct,The passage provided is a positive corporate s...
4,ex_0241,5,5,5,5,correct,The candidate extraction accurately captures t...


In [9]:
comparison.to_csv("task2_genai/results/metrics_comparison.csv")
judge_table.to_csv("task2_genai/results/judge_summary.csv")
base_rows.merge(judge_base, on="id").to_csv("task2_genai/results/base_per_sample.csv", index=False)
ft_rows.merge(judge_ft, on="id").to_csv("task2_genai/results/finetuned_per_sample.csv", index=False)

## Manual review: fine-tuned model (≥ 10 responses)
Each response below shows the passage, reference and prediction, with an *automatic* pre-check (quote grounding + category match). **Read each one yourself** and record your label (`correct`, `partially_correct`, `hallucinated`) in `MANUAL_LABELS`; the auto-label is only a hint.

In [10]:
from task2_genai.evaluation import review_sheet
N_REVIEW = len(test)
sheet = review_sheet(test, [g["text"] for g in ft_gens], ft_rows, N_REVIEW)
for _, r in sheet.iterrows():
    print("=" * 100, f"\n{r.id}  auto={r.auto_label} ({r.auto_reason})\n")
    print("PASSAGE:\n", r.passage, "\n\nREFERENCE:\n", r.gold, "\n\nPREDICTION:\n", r.prediction)

print("\nCopy into the next cell and fill each label with correct / partially_correct / hallucinated:")
print("MANUAL_LABELS = {\n" + "".join(f'    "{i}": "",\n' for i in sheet.id) + "}")

ex_0317  auto=correct (categories match, quotes grounded, severities calibrated)

PASSAGE:
 At Fenwick Holdings, our commitment to operational transparency remains paramount as we navigate an increasingly volatile digital landscape. Our primary cloud infrastructure relies heavily on specialized high-performance chipsets sourced from a single vendor, Aethelgard Silicon, which currently fulfills 92% of our server hardware requirements. Any disruption in their fabrication capacity or geopolitical trade barriers affecting their export licenses could force a total suspension of our core SaaS deployment services. While we previously encountered concerns regarding outdated legacy power cooling in our Nevada data centers, we have successfully retrofitted all facilities with state-of-the-art geothermal heat exchangers, fully neutralizing that efficiency bottleneck. Beyond operational logistics, we are proactively addressing intensifying environmental risks. Our predictive modeling suggests that

In [ ]:
# Manual review: read every response printed above and label it yourself.
# Allowed: "correct", "partially_correct", "hallucinated" (success criteria in Task2_Data.ipynb).
# Self-contained so it can be re-run without the GPU session.
MANUAL_LABELS = {
    "ex_0317": "",
    "ex_0243": "",
    "ex_0042": "",
    "ex_0171": "",
    "ex_0241": "",
    "ex_0312": "",
    "ex_0023": "",
    "ex_0061": "",
    "ex_0273": "",
    "ex_0151": "",
    "ex_0154": "",
    "ex_0019": "",
    "ex_0099": "",
    "ex_0158": "",
    "ex_0324": "",
    "ex_0176": "",
    "ex_0185": "",
    "ex_0326": "",
    "ex_0131": "",
    "ex_0026": "",
    "ex_0294": "",
    "ex_0319": "",
    "ex_0095": "",
    "ex_0257": "",
    "ex_0246": "",
    "ex_0010": "",
    "ex_0147": "",
    "ex_0000": "",
    "ex_0228": "",
    "ex_0163": "",
    "ex_0130": "",
    "ex_0188": "",
}

import os, sys
while not os.path.exists("requirements.txt"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from task2_genai.evaluation import hallucination_rate

ALLOWED = {"correct", "partially_correct", "hallucinated"}
filled = {k: v for k, v in MANUAL_LABELS.items() if v}
bad = {k: v for k, v in filled.items() if v not in ALLOWED}
assert not bad, f"invalid labels: {bad}"
assert len(filled) >= 10, f"label at least 10 responses (have {len(filled)})"
print("MANUAL REVIEW:", hallucination_rate(filled))

## Qualitative analysis

**Where fine-tuning helped.** The base model, with the same system prompt, mostly failed the output contract. Only
12.5% of its responses were valid schema JSON, and category recall was 0.14. It usually returned a partial list or
prose, so its high precision (0.82) only reflects how few categories it committed to. After QLoRA training, every
response was valid JSON (100%), and category F1 rose from 0.24 to 0.94. 90.6% of passages got exactly the right
category set, and 98.5% of evidence quotes were verbatim spans (up from 63.6%). ROUGE-L rose from 0.60 to 0.88 and
BERTScore F1 from 0.87 to 0.98. The LLM judge's "hallucinated" verdicts fell from 37.5% to 3.1%, and its
format-compliance score went from 3.1 to 5.0. For example, ex_0151 extracts three risks across cybersecurity, key
personnel and macroeconomic exposure, each with a verbatim quote and a calibrated severity. ex_0154 recovers all four
risks in a dense airline disclosure.

**Remaining failure modes and next steps.** Most errors sit on taxonomy boundaries rather than in fabricated content.
In ex_0312 and ex_0324, environmental regulation was labelled `esg_climate` instead of `regulatory_legal`. In ex_0000, a
single-manufacturer dependency was read as `customer_concentration` instead of `supply_chain`, and tariffs as
`fx_interest_rate` instead of `geopolitical`. Severity is the weakest dimension (accuracy 0.72); for example, ex_0188
rated a low supply-chain risk as high. The one grounding failure (ex_0154) is a one-word grammatical edit ("discounting"
became "discount") that breaks the verbatim rule. The RAG fallback fired on ex_0000 and fixed the tariff category but
dropped key_personnel, so retrieval alone is not a fix. Next steps:
- Add contrastive training pairs at the confused boundaries.
- Add explicit low/high severity anchors per category.
- Enforce quotes by snapping each one to the closest passage span or by constrained decoding.
- Validate on real 10-K passages to measure synthetic-to-real transfer.

## Bonus: RAG fallback (perplexity-gated ChromaDB retrieval)
Confidence = perplexity of the model's own generated tokens. The threshold is the 80th percentile on the validation set; above it (or on invalid JSON) the two most similar verified training extractions are retrieved from ChromaDB and the model is re-queried with them as calibration context.

In [12]:
from task2_genai.rag_fallback import answer_with_fallback, build_store, calibrate_threshold
from task2_genai.domain import label_response
from task2_genai.evaluation import passage_of
train, val = load_jsonl(DATA_DIR / "train.jsonl"), load_jsonl(DATA_DIR / "val.jsonl")
store = build_store(train)
ft_model, ft_tok = load_for_inference(FT_ID)
val_gens = generate(ft_model, ft_tok, [prompt_messages(r) for r in val])
THRESHOLD = calibrate_threshold([g["perplexity"] for g in val_gens])
print(f"perplexity threshold: {THRESHOLD:.4f}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.6MiB/s]


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

perplexity threshold: 1.0229


In [13]:
ranked = sorted(zip(test, ft_gens), key=lambda x: -x[1]["perplexity"])
cases = [answer_with_fallback(ft_model, ft_tok, rec, store, THRESHOLD) for rec, _ in ranked[:3]]
for rec, case in zip([r for r, _ in ranked[:3]], cases):
    print("=" * 100, f"\n{case['id']} triggered={case['triggered']} reason={case['trigger_reason']}")
    print("BEFORE:", label_response(passage_of(rec), gold_of(rec), case["first"]["text"]), "\n", case["first"]["text"])
    if case["triggered"]:
        print("RETRIEVED:", case["retrieved"])
        print("AFTER: ", label_response(passage_of(rec), gold_of(rec), case["second"]["text"]), "\n", case["second"]["text"])

ex_0000 triggered=True reason=perplexity 1.027 > 1.023
BEFORE: ('partially_correct', "missing=['geopolitical', 'supply_chain'] extra=['customer_concentration', 'fx_interest_rate']") 
 {"risks": [{"category": "customer_concentration", "severity": "medium", "affected_metric": "revenue", "evidence_quote": "our reliance on a single primary apparel manufacturer in Southeast Asia poses a significant risk to inventory availability should production cycles face further disruptions."}, {"category": "fx_interest_rate", "severity": "medium", "affected_metric": "operating margin", "evidence_quote": "rising trade tariffs between regional hubs could materially inflate our landed costs over the next two fiscal years."}, {"category": "key_personnel", "severity": "low", "affected_metric": "market share", "evidence_quote": "We are also monitoring the potential loss of our lead creative director, whose departure would likely destabilize our brand identity."}], "overall_risk_level": "medium"}
RETRIEVED: [